# Phase 5 — NB3: Joint Evaluation (Category → Sentiment)

**Goal:** End-to-end evaluation: Stage 1 predict categories → Stage 2 predict sentiment.
Compare retrieval vs no-retrieval.

**Input:**
- `lcminhc/p5-nb1-stage1` — Stage 1 ckpt + processed data
- `lcminhc/p5-nb2-stage2` — Stage 2 ckpts (retrieval + no-retrieval)
- `lcminhc/p3s2-embedding-flat` — embedding ckpt + FAISS index

**Output:** Metrics report (Category F1, Joint F1, Sentiment Acc|Correct Cat)

## 0. Setup

In [ ]:
!pip install -q transformers faiss-cpu lxml scikit-learn pyyaml

In [ ]:
import os
import sys
import json
import shutil

!git clone https://github.com/lucminhduc3108/Retrieval-ABSA.git /kaggle/working/repo
os.chdir('/kaggle/working/repo')
sys.path.insert(0, '/kaggle/working/repo')
print('Working dir:', os.getcwd())

In [ ]:
# Wire NB1 output
NB1_INPUT = None
for candidate in ['/kaggle/input/p5-nb1-stage1',
                  '/kaggle/input/datasets/lcminhc/p5-nb1-stage1']:
    if os.path.exists(candidate):
        NB1_INPUT = candidate
        break
assert NB1_INPUT, 'Dataset p5-nb1-stage1 not found'

# Wire NB2 output
NB2_INPUT = None
for candidate in ['/kaggle/input/p5-nb2-stage2',
                  '/kaggle/input/datasets/lcminhc/p5-nb2-stage2']:
    if os.path.exists(candidate):
        NB2_INPUT = candidate
        break
assert NB2_INPUT, 'Dataset p5-nb2-stage2 not found'

# Wire embedding + FAISS
EMB_INPUT = None
for candidate in ['/kaggle/input/p3s2-embedding-flat',
                  '/kaggle/input/datasets/lcminhc/p3s2-embedding-flat']:
    if os.path.exists(candidate):
        EMB_INPUT = candidate
        break
assert EMB_INPUT, 'Dataset p3s2-embedding-flat not found'

print(f'NB1: {NB1_INPUT}')
print(f'NB2: {NB2_INPUT}')
print(f'EMB: {EMB_INPUT}')

# Copy processed data
os.makedirs('data/processed', exist_ok=True)
for f in ['category_detection.jsonl', 'sentiment_records.jsonl']:
    shutil.copy(f'{NB1_INPUT}/{f}', f'data/processed/{f}')
    print(f'data/processed/{f}')

# Copy Stage 1 checkpoint
os.makedirs('checkpoints/stage1', exist_ok=True)
shutil.copy(f'{NB1_INPUT}/stage1_best.pt', 'checkpoints/stage1/best.pt')
print(f'Stage 1 ckpt: {os.path.getsize("checkpoints/stage1/best.pt") / 1e6:.1f} MB')

# Copy Stage 2 checkpoints
os.makedirs('checkpoints/stage2', exist_ok=True)
os.makedirs('checkpoints/stage2_noret', exist_ok=True)
shutil.copy(f'{NB2_INPUT}/stage2_best.pt', 'checkpoints/stage2/best.pt')
print(f'Stage 2 (ret) ckpt: {os.path.getsize("checkpoints/stage2/best.pt") / 1e6:.1f} MB')
shutil.copy(f'{NB2_INPUT}/stage2_noret_best.pt', 'checkpoints/stage2_noret/best.pt')
print(f'Stage 2 (no-ret) ckpt: {os.path.getsize("checkpoints/stage2_noret/best.pt") / 1e6:.1f} MB')

# Copy embedding + FAISS
os.makedirs('checkpoints/embedding', exist_ok=True)
shutil.copy(f'{EMB_INPUT}/embedding_best.pt', 'checkpoints/embedding/best.pt')
print(f'Embedding ckpt: {os.path.getsize("checkpoints/embedding/best.pt") / 1e6:.1f} MB')
os.makedirs('indexes', exist_ok=True)
for f in os.listdir(EMB_INPUT):
    if f.endswith(('.faiss', '.npy')) or f == 'train_metadata.jsonl':
        shutil.copy(f'{EMB_INPUT}/{f}', f'indexes/{f}')
        print(f'indexes/{f}')

In [ ]:
import torch
import gc
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
gc.collect()
torch.cuda.empty_cache()

## 1. Evaluate WITH Retrieval

In [ ]:
!python scripts/05_evaluate_joint.py \
    --stage1_ckpt checkpoints/stage1/best.pt \
    --stage2_ckpt checkpoints/stage2/best.pt \
    --embedding_ckpt checkpoints/embedding/best.pt \
    --index_dir indexes/ \
    --stage1_config configs/stage1.yaml \
    --stage2_config configs/stage2.yaml

In [ ]:
# Save retrieval results
if os.path.exists('logs/joint_eval_results.md'):
    shutil.copy('logs/joint_eval_results.md', 'logs/joint_eval_retrieval.md')
    print('Retrieval results saved')

## 2. Evaluate WITHOUT Retrieval

In [ ]:
gc.collect()
torch.cuda.empty_cache()

!python scripts/05_evaluate_joint.py \
    --stage1_ckpt checkpoints/stage1/best.pt \
    --stage2_ckpt checkpoints/stage2_noret/best.pt \
    --stage1_config configs/stage1.yaml \
    --stage2_config configs/stage2_noret.yaml \
    --no_retrieval

In [ ]:
if os.path.exists('logs/joint_eval_results.md'):
    shutil.copy('logs/joint_eval_results.md', 'logs/joint_eval_noret.md')
    print('No-retrieval results saved')

## 3. Results Comparison

In [ ]:
print('=' * 60)
print('RETRIEVAL VARIANT')
print('=' * 60)
if os.path.exists('logs/joint_eval_retrieval.md'):
    with open('logs/joint_eval_retrieval.md') as f:
        print(f.read())

print('\n' + '=' * 60)
print('NO-RETRIEVAL BASELINE')
print('=' * 60)
if os.path.exists('logs/joint_eval_noret.md'):
    with open('logs/joint_eval_noret.md') as f:
        print(f.read())

## 4. Save Outputs

In [ ]:
output_dir = '/kaggle/working/outputs_p5_nb3'
os.makedirs(output_dir, exist_ok=True)

if os.path.exists('logs'):
    shutil.copytree('logs', f'{output_dir}/logs', dirs_exist_ok=True)
    print('logs/ copied')

print(f'\nOutputs saved to {output_dir}')